In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / "api" / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError(
            "Could not locate repo root (expected an `api/` package with pyproject.toml above this notebook)"
        )
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root / "api")

# `Settings` resolves env_file=".env" against the CWD, so the chdir above is what
# makes `settings` work. load_dotenv additionally puts the values in os.environ,
# which is where SDKs (langfuse, huggingface, openai) read them from. Must run
# before importing api.llm, which builds its clients at module level.
env_path = repo_root / "api" / ".env"
loaded = load_dotenv(env_path, override=True)

print("repo root:", repo_root)
print("cwd:", Path.cwd())
print(f"loaded {env_path}:", loaded)
print(os.environ['LANGFUSE_BASE_URL'])

In [ ]:
# Importing this triggers api/llm.py's module-level setup: Langfuse client
# creation + auth_check(), and building the shared `llm` ChatOpenAI client.
from api.document_pipeline.chunking import DocumentChunker
from api.document_pipeline.constants import ChapterInfo,Chunk
from api.llm import llm,langfuse_handler
from api.config import settings
from docling_core.types.doc import DocItemLabel
from docling_core.transforms.chunker.doc_chunk import DocChunk
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from typing import cast


In [ ]:
document_chunker = DocumentChunker()
loaded_document = document_chunker.load_document(Path("./sample_docs/before_the_coffee_gets_cold_split.pdf"))

In [ ]:
chapters_map = {}
last_chapter_count = 0
for page_no in loaded_document.pages:
    for count, (doc_item, level) in enumerate(
        loaded_document.iterate_items(page_no=page_no)
    ):
        if count >= 1:
            break

        if doc_item.label in {
            DocItemLabel.SECTION_HEADER,
            DocItemLabel.TITLE,
        }:
            chapter_info = document_chunker._parse_chapter_heading(doc_item.text)
            if chapter_info.number is None:
                last_chapter_count += 1
                chapter_info.number = last_chapter_count
            else:
                last_chapter_count = chapter_info.number
            chapters_map[doc_item.self_ref] = (doc_item, chapter_info)

In [ ]:
for key,value in chapters_map.items():
  if value[1].is_chapter:
    print(value[1].number,value[1].text,"Label:",value[0].label,value[0].prov[0].page_no)


In [ ]:

EMBEDDING_MODEL_ID = "BAAI/bge-m3"
CACHE_DIR = Path("/home/prinzz/main/my-projects/traverse/api/.cache/")


In [ ]:
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBEDDING_MODEL_ID), max_tokens=1024
)
chunker = HybridChunker(tokenizer=tokenizer)
chunks: list[DocChunk] = cast(
    list[DocChunk], list(chunker.chunk(dl_doc=loaded_document))
)
model = SentenceTransformer(EMBEDDING_MODEL_ID, device="cuda",               cache_folder=str(CACHE_DIR), local_files_only=True)


In [ ]:
contextualized_texts = [chunker.contextualize(chunk) for chunk in chunks]
embeddings = model.encode(
    contextualized_texts,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

result_chunks = []


In [ ]:
chunks[0].meta.doc_items

In [ ]:
import httpx

LLM_MODEL_ID = "Qwen/Qwen3-8B-AWQ"
LLM_BASE_URL = "http://localhost:8080/v1/"

# The chunks above were sized with the bge-m3 tokenizer (max_tokens=1024) because
# that's what embeds them. Batching for the extraction call has to be measured
# with the LLM's own tokenizer instead - different vocab, so the same text is a
# different number of tokens.
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)


def get_max_model_len(default: int = 40960) -> int:
    """Read the served context length from vLLM, falling back to the config value.

    `default` is Qwen3-8B's max_position_embeddings. vLLM is usually started
    with a smaller --max-model-len to fit the KV cache in VRAM, so the live
    value is the one to trust whenever the server is reachable.
    """
    try:
        response = httpx.get(f"{LLM_BASE_URL}models", timeout=5.0)
        response.raise_for_status()
        return int(response.json()["data"][0]["max_model_len"])
    except Exception as error:
        print(f"could not reach {LLM_BASE_URL}models ({error}); assuming {default}")
        return default


def count_tokens(text: str) -> int:
    """Raw token count for a piece of text, no chat-template scaffolding."""
    return len(llm_tokenizer.encode(text, add_special_tokens=False))


def count_prompt_tokens(prompt: str) -> int:
    """Token count for a system prompt as the server will actually see it."""
    return len(
        llm_tokenizer.apply_chat_template(
            [{"role": "system", "content": prompt}],
            add_generation_prompt=True,
            tokenize=True,
        )
    )


MAX_MODEL_LEN = get_max_model_len()
print("max model len:", MAX_MODEL_LEN)

chunk_token_counts = [count_tokens(chunk.text) for chunk in chunks]
print(
    f"chunks: {len(chunk_token_counts)}, "
    f"tokens min/mean/max: {min(chunk_token_counts)}/"
    f"{sum(chunk_token_counts) // len(chunk_token_counts)}/{max(chunk_token_counts)}"
)

In [ ]:
from langchain.messages import SystemMessage
from pydantic import BaseModel, Field

class ExtractedCharacter(BaseModel):
    name: str
    aliases: list[str] = Field(default_factory=list)
    has_appeared_earlier: bool


class CharacterExtractionOutput(BaseModel):
    characters: list[ExtractedCharacter]


def build_character_extraction_prompt(
    current_chunks: list[DocChunk],
    previous_chunks: list[DocChunk],
    previous_characters: list[ExtractedCharacter],
) -> str:
    known_characters = ""
    if len(previous_characters) > 0:
        known_characters = (
            "\n".join(
                f"- {character.name}"
                + (
                    f" (also called: {', '.join(character.aliases)})"
                    if character.aliases
                    else ""
                )
                for character in previous_characters
            )
            or "(none yet)"
        )
    preceding_context = (
        "\n\n".join(previous_chunk.text for previous_chunk in previous_chunks)
        or "(none, this is the start of the book)"
    )
    # A batch can span several docling chunks. Join them with a visible
    # boundary marker so the model still reads it as one continuous span
    # rather than several unrelated fragments.
    current_text = (
        "\n\n[... chunk boundary ...]\n\n".join(
            chunk.text for chunk in current_chunks
        )
        or "(empty)"
    )

    return f"""You are extracting the characters that appear in a span of a novel.
            You are given three things:
            1. KNOWN CHARACTERS - characters already extracted from earlier parts of the book.
            2. PRECEDING CONTEXT - the text immediately before the current span. Use it ONLY to resolve pronouns and partial names; never extract a character who appears only there.
            3. CURRENT SPAN - the text you must extract from. It may be made of several chunks joined by "[... chunk boundary ...]"; treat it as one continuous span.

            Rules:
            - Return every person who appears in the CURRENT SPAN: anyone who speaks, acts, is addressed, or is referred to by name.
            - `name`: the character's canonical name, the fullest form used for them so far. Reuse the exact spelling from KNOWN CHARACTERS when it is the same person. Strip honorifics, titles and terms of endearment ("Kazu, dear" -> "Kazu", "Miss Fumiko" -> "Fumiko").
            - `aliases`: other forms used for that same person in this span (nickname, surname only, given name only, an epithet such as "the waitress"). Empty list if there are none.
            - `has_appeared_earlier`: true if the character is in KNOWN CHARACTERS or clearly appears in PRECEDING CONTEXT; false if this span is their first appearance in the book.
            - Merge every reference to one person into a single entry. Never output the same person twice, and never output an alias as its own character.
            - If someone is referred to only by a pronoun but the preceding context makes their identity unambiguous, record them under their canonical name.
            - Do NOT extract places, cafes, organisations, objects, book or chapter titles, the author's or translator's name, or generic groups ("the customers", "the family").
            - Do NOT extract an unnamed background person ("a man at the counter") unless the narrative treats them as an actual participant in the scene.
            - If the span contains no characters at all (front matter, a chapter heading, a copyright page), return an empty list.

            KNOWN CHARACTERS:
            {known_characters}

            PRECEDING CONTEXT (reference only, do not extract from it):
            {preceding_context}

            CURRENT SPAN:
            {current_text}
    """


def extract_characters_from_chunks(
    current_chunks: list[DocChunk],
    previous_chunks: list[DocChunk],
    previous_characters: list[ExtractedCharacter],
) -> CharacterExtractionOutput:
    prompt = build_character_extraction_prompt(
        current_chunks, previous_chunks, previous_characters
    )

    structured_llm = llm.with_structured_output(CharacterExtractionOutput)
    result = structured_llm.invoke(
        [SystemMessage(prompt)], config={"callbacks": [langfuse_handler]}
    )

    return cast(CharacterExtractionOutput, result)


def merge_extracted_characters(
    known: list[ExtractedCharacter], newly_found: list[ExtractedCharacter]
) -> list[ExtractedCharacter]:
    """Fold one batch's characters into the running list, deduping by name.

    The prompt asks the model to reuse existing spellings, but nothing stops
    a later batch spelling/casing a name slightly differently - matching
    case-insensitively is cheap insurance against duplicate character nodes
    once this feeds the knowledge graph.
    """
    by_name = {character.name.lower(): character for character in known}

    for character in newly_found:
        key = character.name.lower()
        existing = by_name.get(key)

        if existing is None:
            by_name[key] = character
            continue

        merged_aliases = list(dict.fromkeys(existing.aliases + character.aliases))
        by_name[key] = existing.model_copy(
            update={
                "aliases": merged_aliases,
                "has_appeared_earlier": existing.has_appeared_earlier
                or character.has_appeared_earlier,
            }
        )

    return list(by_name.values())

In [ ]:
BATCH_OUTPUT_RESERVE_TOKENS = 2048  # room for the structured-output response
PRECEDING_CONTEXT_WINDOW = 5  # lookback chunks per batch, matches the original per-chunk loop


def _prompt_overhead_tokens(batch_start_index: int) -> int:
    """Token cost of everything in the prompt except the current span's text.

    Recomputed at each batch boundary since preceding context shifts with
    position in the book. Growth of KNOWN CHARACTERS over the course of the
    run isn't modeled exactly here - it stays small next to
    BATCH_OUTPUT_RESERVE_TOKENS, which absorbs it.
    """
    preceding = chunks[max(0, batch_start_index - PRECEDING_CONTEXT_WINDOW):batch_start_index]
    shell_prompt = build_character_extraction_prompt(
        current_chunks=[], previous_chunks=preceding, previous_characters=[]
    )
    return count_prompt_tokens(shell_prompt)


def build_chunk_batches(chunks: list[DocChunk]) -> list[list[int]]:
    """Group chunk indices into batches whose combined prompt fits MAX_MODEL_LEN.

    Chunks are packed greedily in document order and never split across
    batches. A single chunk too large for the remaining budget still gets
    its own (oversized) batch rather than being silently truncated - that
    surfaces as an over-length error from the server instead of quietly
    dropping text.
    """
    batches: list[list[int]] = []
    batch: list[int] = []
    batch_tokens = 0
    budget = 0

    for index, chunk in enumerate(chunks):
        if not batch:
            overhead = _prompt_overhead_tokens(index)
            budget = MAX_MODEL_LEN - overhead - BATCH_OUTPUT_RESERVE_TOKENS
            if budget <= 0:
                raise ValueError(
                    f"Prompt overhead ({overhead}) plus output reserve "
                    f"({BATCH_OUTPUT_RESERVE_TOKENS}) already exceeds "
                    f"MAX_MODEL_LEN ({MAX_MODEL_LEN})."
                )

        chunk_tokens = chunk_token_counts[index]

        if batch and batch_tokens + chunk_tokens > budget:
            batches.append(batch)
            batch = []
            batch_tokens = 0
            overhead = _prompt_overhead_tokens(index)
            budget = MAX_MODEL_LEN - overhead - BATCH_OUTPUT_RESERVE_TOKENS

        batch.append(index)
        batch_tokens += chunk_tokens

    if batch:
        batches.append(batch)

    return batches


chunk_batches = build_chunk_batches(chunks)
batch_sizes = [len(batch) for batch in chunk_batches]
print(
    f"{len(chunks)} chunks packed into {len(chunk_batches)} batches "
    f"(chunks/batch min/mean/max: {min(batch_sizes)}/"
    f"{sum(batch_sizes) // len(batch_sizes)}/{max(batch_sizes)})"
)

In [ ]:
previous_chapter = ChapterInfo(is_chapter=False)

previous_characters: list[ExtractedCharacter] = []
batch_end_index = {batch[-1] for batch in chunk_batches}
current_llm_batch: list[DocChunk] = []
current_batch_start = 0

for index, (chunk, embedding) in enumerate(zip(chunks, embeddings, strict=True)):
    # Map chapter to chunk.
    for heading in chunk.meta.headings:
        for value in chapters_map.values():
            if value[1].text == heading and heading != previous_chapter.text:
                previous_chapter = value[1]

    # Extract characters once a batch closes (batches built above from the
    # LLM's own tokenizer + MAX_MODEL_LEN, not the embedding chunk size).
    current_llm_batch.append(chunk)
    if index in batch_end_index:
        preceding_chunks = chunks[max(0, current_batch_start - PRECEDING_CONTEXT_WINDOW):current_batch_start]
        extraction = extract_characters_from_chunks(
            current_llm_batch, preceding_chunks, previous_characters
        )
        previous_characters = merge_extracted_characters(
            previous_characters, extraction.characters
        )
        current_llm_batch = []
        current_batch_start = index + 1

    # Provenance build.
    chunk_provenance_list = [
        provenance
        for doc_item in chunk.meta.doc_items
        for provenance in doc_item.prov
    ]

    pages = sorted([provenance.page_no for provenance in chunk_provenance_list])

    result_chunks.append(
        Chunk(
            text_embedding=embedding.tolist(),
            text=chunk.text,
            pages=pages,
            page_start=pages[0] if pages else 0,
            page_end=pages[-1] if pages else 0,
            chapter=previous_chapter,
        )
    )


In [ ]:
print(previous_characters)

In [ ]:
items = [{"text": chunk.text[:20], "chapter": f'{chunk.chapter.number} {chunk.chapter.title}'} for chunk in result_chunks if chunk.chapter.is_chapter]

In [ ]:
for i in items:
  print(i)